# 32 - Base vs Fine-Tuned RAG Semantic Similarity

This notebook computes answer-level semantic similarity for the existing base RAG and fine-tuned RAG outputs. It does not use the external-mapped experiments.


In [ ]:
!python -m pip install -q -U "sentence-transformers>=5.1.0" "transformers>=4.51.0" accelerate pandas numpy


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import sys
import json

DRIVE_ROOT = Path('/content/drive/MyDrive/TURKISH_LEGAL_RAG')
os.chdir(DRIVE_ROOT)
if str(DRIVE_ROOT) not in sys.path:
    sys.path.insert(0, str(DRIVE_ROOT))

print('Working directory:', Path.cwd())


In [ ]:
from src.evaluate_answer_semantic_similarity import evaluate_semantic_similarity

output_dir = DRIVE_ROOT / 'outputs/generation_eval'
model_name = 'Qwen/Qwen3-Embedding-8B'
device = 'cuda'

base_predictions = output_dir / 'qwen3_32b_base_rag_predictions_v1.csv'
finetuned_predictions = output_dir / 'qwen3_32b_finetuned_rag_predictions_v1.csv'

for required in [base_predictions, finetuned_predictions]:
    if not required.exists():
        raise FileNotFoundError(required)

print('Base predictions:', base_predictions)
print('Fine-tuned predictions:', finetuned_predictions)


In [ ]:
base_semantic = evaluate_semantic_similarity(
    predictions_csv=base_predictions,
    output_eval_csv=output_dir / 'qwen3_32b_base_rag_semantic_eval_v1.csv',
    output_summary_json=output_dir / 'qwen3_32b_base_rag_semantic_summary_v1.json',
    model_name=model_name,
    batch_size=4,
    device=device,
)
print(json.dumps(base_semantic['metrics'], ensure_ascii=False, indent=2))


In [ ]:
finetuned_semantic = evaluate_semantic_similarity(
    predictions_csv=finetuned_predictions,
    output_eval_csv=output_dir / 'qwen3_32b_finetuned_rag_semantic_eval_v1.csv',
    output_summary_json=output_dir / 'qwen3_32b_finetuned_rag_semantic_summary_v1.json',
    model_name=model_name,
    batch_size=4,
    device=device,
)
print(json.dumps(finetuned_semantic['metrics'], ensure_ascii=False, indent=2))


In [ ]:
import pandas as pd

comparison = pd.DataFrame([
    {'system': 'Base RAG', **base_semantic['metrics']},
    {'system': 'Fine-tuned RAG', **finetuned_semantic['metrics']},
])
comparison_path = output_dir / 'base_vs_finetuned_rag_semantic_comparison_v1.csv'
comparison.to_csv(comparison_path, index=False, encoding='utf-8-sig')
display(comparison)
print('Saved:', comparison_path)
